# DST Domestic Workflow v1.2 R1_07_21 — Complex Transformations
PySpark notebook for transformations migrated from Alteryx.
Handles: window functions, pivots, and advanced aggregations.

In [ ]:
# Pipeline parameters (injected by Fabric pipeline)
fiscal_year = ""  # e.g., "2025"
fiscal_period = ""  # e.g., "01"
run_id = ""  # Pipeline run ID

StatementMeta(, 942bec23-f47c-43ae-860e-23fcb4ab05d7, 9, Finished, Available, Finished, False)

In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from datetime import datetime

spark = SparkSession.builder.getOrCreate()
print(f"Spark session initialized. Run ID: {run_id}")

StatementMeta(, 942bec23-f47c-43ae-860e-23fcb4ab05d7, 10, Finished, Available, Finished, False)

Spark session initialized. Run ID: 


In [ ]:
# Read from bronze tables
# Lakehouse: DST Domestic Workflow v1.2 R1_07_21_lakehouse
WORKSPACE_ID = "c22e4776-e91c-4f5a-9118-f8049b2c5c5f"
LAKEHOUSE_ID  = "4fa33def-575d-481c-a019-139a51fc9ba0"
BASE = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}/Tables"

bronze_Data_Hub_Schema_290 = spark.read.format("delta").load(f"{BASE}/bronze_Data_Hub_Schema_290")
bronze_source_66 = spark.read.format("delta").load(f"{BASE}/bronze_source_66")
bronze_source_251 = spark.read.format("delta").load(f"{BASE}/bronze_source_251")
bronze_source_254 = spark.read.format("delta").load(f"{BASE}/bronze_source_254")
bronze_source_296 = spark.read.format("delta").load(f"{BASE}/bronze_source_296")
bronze_Get_Snowflake_Data_39 = spark.read.format("delta").load(f"{BASE}/bronze_Get_Snowflake_Data_39")
bronze_Collection_ID_312 = spark.read.format("delta").load(f"{BASE}/bronze_Collection_ID_312")
print("Bronze tables loaded successfully.")

StatementMeta(, 942bec23-f47c-43ae-860e-23fcb4ab05d7, 11, Finished, Available, Finished, False)

Bronze tables loaded successfully.


## Window Functions
Equivalent to Alteryx Multi-Row Formula tools.
These perform ordered calculations within partitions.

In [ ]:
# Build lookup dict of all bronze tables
bronze_dfs = {
    "bronze_Data_Hub_Schema_290":   bronze_Data_Hub_Schema_290,
    "bronze_source_66":             bronze_source_66,
    "bronze_source_251":            bronze_source_251,
    "bronze_source_254":            bronze_source_254,
    "bronze_source_296":            bronze_source_296,
    "bronze_Get_Snowflake_Data_39": bronze_Get_Snowflake_Data_39,
    "bronze_Collection_ID_312":     bronze_Collection_ID_312,
}

# Auto-detect which bronze table has impression + entity data
df_impressions_source = None
df_amounts_source     = None

for name, df in bronze_dfs.items():
    cols_lower = {c.lower() for c in df.columns}
    if "impressions" in cols_lower and "mapped entity" in cols_lower:
        df_impressions_source = df
        df_amounts_source     = df  # same table also has Amount in Group Curr_
        print(f"Impressions / amounts source: {name} ({len(df.columns)} cols): {df.columns}")

if df_impressions_source is None:
    print("WARNING: Could not auto-detect impressions source. Printing schemas:")
    for name, df in bronze_dfs.items():
        print(f"  {name}: {df.columns}")
else:
    # T221 — impression-level aggregation
    df_impressions = df_impressions_source.groupBy(
        "Month and Year", "Ad Unit", "Mapped Entity",
        "Cleansed_State", "Country", "City", "Application"
    ).agg(
        F.sum("Impressions").alias("Sum_Impressions")
    )
    print(f"T221 impression aggregation: {df_impressions.count()} rows")

    # T220 — amount aggregation (column name uses underscore, not dot)
    df_amounts = df_amounts_source.groupBy(
        "Month and Year", "Mapped Entity"
    ).agg(
        F.sum("Amount in Group Curr_").alias("Total_Amount")
    )
    print(f"T220 amount aggregation: {df_amounts.count()} rows")

    # T169 — denominator without NA
    df_denom_without_na = df_impressions_source.filter(
        F.col("Impressions").isNotNull()
    ).groupBy(
        "Mapped Entity", "Month and Year"
    ).agg(
        F.sum("Impressions").alias("Denominator_without_NA")
    )

    # Denominator with NA (all rows)
    df_denom_with_na = df_impressions_source.groupBy(
        "Mapped Entity", "Month and Year"
    ).agg(
        F.sum("Impressions").alias("Denominator_with_NA")
    )
    print("Denominator aggregations complete.")

StatementMeta(, 942bec23-f47c-43ae-860e-23fcb4ab05d7, 12, Finished, Available, Finished, False)

Impressions / amounts source: bronze_Data_Hub_Schema_290 (20 cols): ['Month and Year', 'Mapped Entity', 'Amount in Group Curr_', 'Ad Unit', 'Cleansed_State', 'Impressions', 'Denominator without NA', 'Denominator with NA', 'Percent without NA', 'Percent with NA', 'Year', 'Month', 'Apportioned Amount without NA', 'Apportioned Amount with NA', 'Country', 'City', 'Application', 'Placeholder 1', 'Placeholder 2', 'Placeholder 3']
T221 impression aggregation: 20 rows
T220 amount aggregation: 20 rows
Denominator aggregations complete.


## Pivot Operations
Equivalent to Alteryx Cross Tab tools.

In [ ]:
# T239/T246 — percentage and apportioned amount calculations
# Depends on Summarize results from cell 6
if df_impressions_source is None:
    print("Skipping: impressions source not found — check cell 6.")
else:
    join_keys = ["Mapped Entity", "Month and Year"]

    df_with_denoms = (
        df_impressions
        .join(df_denom_without_na, join_keys, "left")
        .join(df_denom_with_na,    join_keys, "left")
    )

    # T239 — calculate impression percentages
    df_pct = df_with_denoms.withColumn(
        "Percent without NA",
        F.col("Sum_Impressions") / F.col("Denominator_without_NA")
    ).withColumn(
        "Percent with NA",
        F.col("Sum_Impressions") / F.col("Denominator_with_NA")
    )

    # T246 — calculate apportioned amounts
    df_result = df_pct.join(
        df_amounts, join_keys, "left"
    ).withColumn(
        "Apportioned Amount without NA",
        -F.round(F.col("Total_Amount") * F.col("Percent without NA"), 2)
    ).withColumn(
        "Apportioned Amount with NA",
        -F.round(F.col("Total_Amount") * F.col("Percent with NA"), 2)
    )
    print(f"Formula transformations complete. df_result row count: {df_result.count()}")
    df_result.printSchema()

StatementMeta(, 942bec23-f47c-43ae-860e-23fcb4ab05d7, 13, Finished, Available, Finished, False)

Formula transformations complete. df_result row count: 20
root
 |-- Mapped Entity: string (nullable = true)
 |-- Month and Year: string (nullable = true)
 |-- Ad Unit: string (nullable = true)
 |-- Cleansed_State: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Application: string (nullable = true)
 |-- Sum_Impressions: double (nullable = true)
 |-- Denominator_without_NA: double (nullable = true)
 |-- Denominator_with_NA: double (nullable = true)
 |-- Percent without NA: double (nullable = true)
 |-- Percent with NA: double (nullable = true)
 |-- Total_Amount: double (nullable = true)
 |-- Apportioned Amount without NA: double (nullable = true)
 |-- Apportioned Amount with NA: double (nullable = true)



In [ ]:
# Write transformed data to silver table via abfss (no default lakehouse needed)
import re

if "df_result" in dir() and df_result is not None:
    # Sanitize column names: replace invalid Delta chars with underscores
    def safe_col(name):
        invalid = list(" ,;{}()=")
        result = name
        for c in invalid:
            result = result.replace(c, "_")
        return result.strip("_")

    safe_cols = [safe_col(c) for c in df_result.columns]
    df_out = df_result.toDF(*safe_cols)

    silver_path = f"{BASE}/silver_transformed"
    df_out.write.format("delta").mode("overwrite").save(silver_path)
    print(f"Written to silver_transformed at {silver_path}")
    print(f"Columns: {df_out.columns}")
    print(f"Row count: {df_out.count()}")
else:
    print("Skipping write: df_result not defined. Complete cells 6 and 8 first.")

print(f"Notebook execution complete. Run ID: {run_id}")
print(f"Fiscal Year: {fiscal_year}, Period: {fiscal_period}")

StatementMeta(, 942bec23-f47c-43ae-860e-23fcb4ab05d7, 14, Finished, Available, Finished, False)

Written to silver_transformed at abfss://c22e4776-e91c-4f5a-9118-f8049b2c5c5f@onelake.dfs.fabric.microsoft.com/4fa33def-575d-481c-a019-139a51fc9ba0/Tables/silver_transformed
Columns: ['Mapped_Entity', 'Month_and_Year', 'Ad_Unit', 'Cleansed_State', 'Country', 'City', 'Application', 'Sum_Impressions', 'Denominator_without_NA', 'Denominator_with_NA', 'Percent_without_NA', 'Percent_with_NA', 'Total_Amount', 'Apportioned_Amount_without_NA', 'Apportioned_Amount_with_NA']
Row count: 20
Notebook execution complete. Run ID: 
Fiscal Year: , Period: 
